In [ ]:
import os
import json
import yaml
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from termcolor import colored, cprint

from inspect_ai.log import read_eval_log

In [ ]:
model_info = pd.read_csv("../data/models_pricing.csv")

In [ ]:
root_dir = "../outputs/first_epoch"
eval_dir = "text"

In [ ]:
def clean_model_name(model_name: str) -> str:
	if "google/" in model_name:
		out =  model_name.split("/")[-1].strip()
		out = out.replace("-preview", "")
		return out
	elif "openai/" in model_name:
		if "gpt" in model_name or "/o3" in model_name:
			return model_name.split("/")[-1].strip()
		else:
			return model_name.split("openai/")[-1].split("/")[-1].strip().replace("thinking", "").replace("Thinking", "").strip(" -")
	else:
		return model_name
	
def get_model_family(model_name: str) -> tuple:
	"""
	Returns model family name + version number
	"""
	if "gpt-oss" in model_name:
		return "GPT-OSS", None
	elif "Kimi-" in model_name:
		return "Kimi", None
	elif "GLM" in model_name:
		return "GLM", None
	elif "gemini" in model_name:
		return "Gemini", 3 if "3" in model_name else ("2.5" if "2.5" in model_name else model_name.split("gemini-")[-1].split("-")[0])
	elif "gpt" in model_name:
		return "GPT", model_name.split("gpt-")[-1].split("-")[0]
	elif "gemma" in model_name:
		return "Gemma", model_name.split("gemma-")[-1].split("-")[0]
	elif "Qwen" in model_name:
		return "Qwen", model_name.split("Qwen-")[-1].split("-")[0]
	elif "deepseek" in model_name.lower():
		return "DeepSeek", None
	else:
		return model_name, None

In [ ]:

results = []
for model_dir in os.listdir(os.path.join(root_dir, eval_dir)):
    model_path = os.path.join(root_dir, eval_dir, model_dir)
    print("Processing: " + colored(model_dir, "cyan"))

    cfg = yaml.safe_load(open(os.path.join(model_path, ".hydra","config.yaml"), "r"))
    model_name = cfg["solver"]["model_name"]
    

    # Get model info
    input_price_per_1M = model_info[model_info["model_name"] == model_name]["input_price"].values[0]
    output_price_per_1M = model_info[model_info["model_name"] == model_name]["output_price"].values[0]
    release_date = model_info[model_info["model_name"] == model_name]["release_date"].values[0]
    size = model_info[model_info["model_name"] == model_name]["size"].values[0]
    model_type = model_info[model_info["model_name"] == model_name]["model_type"].values[0]
    model_family, model_version = get_model_family(model_name)
    
    # Read evaluation log
    try:
        eval_files = [f for f in os.listdir(model_path) if f.endswith(".eval")]
        log_file = os.path.join(model_path, eval_files[0])
        log = read_eval_log(log_file)
    except Exception as e:
        print(colored(f"\tError reading log for {model_dir}: {e}", "red"))
        continue

    n_models = len(log.stats.model_usage.keys())
    

    if n_models > 1:
        d = log.stats.model_usage.copy()
        d.pop("google/gemini-3-flash-preview", None)    # Remove grader usage
        model_name = clean_model_name(list(d.keys())[0])

        if "gpt-5" in model_name.lower() or "o3" in model_name.lower():
            # FIXME: double check whether this is still the case or not
            in_tks = out_tks = reas_tks = 0
            for sample in log.samples:
                if sample.error:
                    continue
                usage = sample.model_usage[list(d.keys())[0]]
                in_tks += usage.input_tokens
                out_tks += usage.output_tokens
        else:
            usage = list(d.values())[0]
            in_tks, out_tks, reas_tks = usage.input_tokens, usage.output_tokens, usage.reasoning_tokens
            reas_tks = 0 if reas_tks is None else reas_tks


        input_cost = (in_tks / 1_000_000) * input_price_per_1M
        output_cost = ((out_tks + reas_tks) / 1_000_000) * output_price_per_1M
        total_cost = input_cost + output_cost


    elif n_models == 1:
        d = log.stats.model_usage.copy()
        model_name = clean_model_name(list(d.keys())[0])

        # Compute cost for grading only
        gr_in_tks = 0
        gr_out_tks = 0
        gr_reas_tks = 0

        gr_usage = log.stats.role_usage.get("grader", None)
        sv_usage = log.stats.role_usage.get("solver", None)
        print(f"\tGrader usage: {gr_usage}")

        in_tks = sv_usage.input_tokens - gr_in_tks
        out_tks = sv_usage.output_tokens - gr_out_tks
        reas_tks = sv_usage.reasoning_tokens - gr_reas_tks
        reas_tks = 0 if reas_tks is None else reas_tks
        
        input_cost = (in_tks / 1_000_000) * input_price_per_1M
        output_cost = ((out_tks + reas_tks) / 1_000_000) * output_price_per_1M
        total_cost = input_cost + output_cost
    else:
        print(colored(f"\tError: No model usage found in {log_file}", "red"))
        continue

    # Compute number valid samples
    num_valid = sum(1 for sample in log.samples if sample.score is not None and sample.error is None)
    num_correct = sum(1 for sample in log.samples if sample.score is not None and sample.score.value in [1, True, "C"] and sample.error is None)
    num_total_samples = len(log.samples)


    # TODO: Gather multiple epochs
    # scores_by_id = {}
    # for sample in log.samples:
    #     if sample.id not in scores_by_id:
    #         scores_by_id[sample.id] = []
    #     scores_by_id[sample.id].append(1 if sample.score is not None and sample.score.value in [1, True, "C"] else 0)
    

    results.append({
        "experiment": root_dir.split("/")[-1],
        "eval_dir": eval_dir,
        "model_name": model_name,
        "model_family": model_family,
        "model_version": model_version,
        "release_date": release_date,
        "size": size,
        "model_type": model_type,
        "accuracy": num_correct / num_total_samples,
        "num_samples": num_total_samples,
        "num_valid_samples": num_valid,
        "num_correct_samples": num_correct,
        "accuracy": num_correct / num_total_samples if num_total_samples > 0 else 0,
        "accuracy_valid": num_correct / num_valid if num_valid > 0 else 0,
        "input_tokens": in_tks,
        "output_tokens": out_tks + reas_tks,
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
        "avg_price_per_sample": total_cost / num_valid if num_valid > 0 else 0,  # average only over valid samples
        "avg_out_tokens_per_sample": (out_tks + reas_tks) / num_valid if num_valid > 0 else 0,
    })



## Create table

In [ ]:
# TODO:

---

## Create plot

In [ ]:
df = pd.DataFrame(results)
df[["model_name", "avg_price_per_sample", "accuracy"]].sort_values("accuracy", ascending=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# ------------------------------------------------------------------
# CLEANING / PREP
# ------------------------------------------------------------------
plot_df = df.copy()

plot_df["accuracy_pct"] = plot_df["accuracy"] * 100
plot_df["avg_price_per_100_samples"] = plot_df["avg_price_per_sample"] * 100

# Group key = manufacturer/family + version
plot_df["series"] = (
    plot_df["model_family"].astype(str)
    + " "
    + plot_df["model_version"].astype(str)
)

plot_df = plot_df.dropna(
    subset=["accuracy_pct", "avg_price_per_100_samples"]
).copy()

# Sort lines by model size if available, otherwise by cost
plot_df["sort_key"] = np.where(
    plot_df["size"].notna(),
    plot_df["size"],
    plot_df["avg_price_per_100_samples"]
)

# ------------------------------------------------------------------
# STYLING
# ------------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(16, 10))

families = plot_df["model_family"].dropna().unique()
palette = sns.color_palette("tab10", n_colors=len(families))
family_colors = dict(zip(families, palette))

# ------------------------------------------------------------------
# DRAW CONNECTED SERIES
# ------------------------------------------------------------------
for series_name, g in plot_df.groupby("series"):
    g = g.sort_values("sort_key")

    fam = g["model_family"].iloc[0]
    color = family_colors.get(fam, "gray")

    # connected line
    ax.plot(
        g["avg_price_per_100_samples"],
        g["accuracy_pct"],
        lw=1.8,
        alpha=0.8,
        color=color,
        zorder=1
    )

    # scatter points
    ax.scatter(
        g["avg_price_per_100_samples"],
        g["accuracy_pct"],
        s=60,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.95,
        zorder=2
    )

# ------------------------------------------------------------------
# LABEL BEST POINT OF EACH SERIES (optional)
# ------------------------------------------------------------------
for series_name, g in plot_df.groupby("series"):
    best = g.loc[g["accuracy_pct"].idxmax()]

    ax.annotate(
        best["model_name"],
        xy=(best["avg_price_per_100_samples"], best["accuracy_pct"]),
        xytext=(8, 0),
        textcoords="offset points",
        fontsize=9,
        alpha=0.9
    )

# ------------------------------------------------------------------
# AXES
# ------------------------------------------------------------------
ax.set_xscale("log")

ax.set_xlabel("Average Cost per 100 Samples ($)", fontsize=14)
ax.set_ylabel("Accuracy (%)", fontsize=14)
ax.set_title(
    f"{eval_dir.upper()} - Model Performance vs Cost",
    fontsize=18,
    weight="bold",
    pad=20
)

# ax.set_ylim(0, 100)

# Format x-axis as dollars
ax.xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f"${x:.4f}" if x < 0.01 else f"${x:.2f}")
)

# ------------------------------------------------------------------
# LEGEND (one entry per family)
# ------------------------------------------------------------------
handles = []
labels = []

for fam in families:
    h = plt.Line2D(
        [0], [0],
        color=family_colors[fam],
        marker="o",
        lw=2,
        markersize=7
    )
    handles.append(h)
    labels.append(fam)

ax.legend(
    handles,
    labels,
    title="Model Family",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

# ------------------------------------------------------------------
# GRID / LAYOUT
# ------------------------------------------------------------------
ax.grid(True, which="major", alpha=0.25)
ax.grid(True, which="minor", alpha=0.08)

plt.tight_layout()
plt.show()

In [ ]:
import requests
import pandas as pd

aai_index_path = "../data/artificial_analysis_intelligence_index.csv"

if not os.path.exists(aai_index_path):
    API_KEY = os.getenv("ARTIFICIAL_ANALYSIS_API_KEY")

    url = "https://artificialanalysis.ai/api/v2/data/llms/models"
    headers = {"x-api-key": API_KEY}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    models = r.json()["data"]

    rows = []
    for m in models:
        evals = m.get("evaluations", {})

        rows.append({
            "model_id": m.get("id"),
            "model_name": m.get("name"),
            "creator": m.get("model_creator", {}).get("name"),
            "aa_intelligence_index": evals.get("artificial_analysis_intelligence_index"),
            "aa_coding_index": evals.get("artificial_analysis_coding_index"),
            "aa_math_index": evals.get("artificial_analysis_math_index"),
        })

    aai_index = pd.DataFrame(rows)
    aai_index.to_csv(aai_index_path, index=False)
else:
    aai_index = pd.read_csv(aai_index_path)
    aai_index_dict = {}
    for _, row in aai_index.iterrows():
        aai_index_dict[row["model_name"]] = row["aa_intelligence_index"]

In [ ]:
mapping = {
    # OpenAI reasoning models, medium effort where available
    "gpt-5.4-mini": aai_index_dict.get("GPT-5.4 mini (medium)", None),
    "gpt-5.4-nano": aai_index_dict.get("GPT-5.4 nano (medium)", None),
    "gpt-5.2": aai_index_dict.get("GPT-5.2 (medium)", None),
    "gpt-5": aai_index_dict.get("GPT-5 (medium)", None),
    "gpt-5-mini": aai_index_dict.get("GPT-5 mini (medium)", None),
	# missing values, use proxy
    "gpt-5.4": (aai_index_dict["GPT-5.4 (xhigh)"] + aai_index_dict["GPT-5.4 (Non-reasoning)"]) / 2, # proxy for medium effort
    "gpt-oss-120b": (aai_index_dict["gpt-oss-120B (high)"] + aai_index_dict["gpt-oss-120B (low)"]) / 2, # proxy for medium effort,
    "gpt-oss-20b": (aai_index_dict["gpt-oss-20B (high)"] + aai_index_dict["gpt-oss-20B (low)"]) / 2, # proxy for medium effort,

    # Google / Gemini / Gemma
    "gemini-3.1-pro": aai_index_dict.get("Gemini 3.1 Pro Preview", None),
    "gemini-3-flash": aai_index_dict.get("Gemini 3 Flash Preview (Reasoning)", None),
    "gemini-3.1-flash-lite": aai_index_dict.get("Gemini 3.1 Flash-Lite Preview", None),
    "gemini-2.5-pro": aai_index_dict.get("Gemini 2.5 Pro", None),
    "gemini-2.5-flash": aai_index_dict.get("Gemini 2.5 Flash (Reasoning)", None),
    "gemma-4-31B-it": aai_index_dict.get("Gemma 4 31B (Reasoning)", None),
    "gemma-4-26B-A4B-it": aai_index_dict.get("Gemma 4 26B A4B (Reasoning)", None),
    "gemma-4-E4B-it": aai_index_dict.get("Gemma 4 E4B (Reasoning)", None),
    "gemma-4-E2B-it": aai_index_dict.get("Gemma 4 E2B (Reasoning)", None),

    # Qwen reasoning models
    "Qwen3.5-397B-A17B": aai_index_dict.get("Qwen3.5 397B A17B (Reasoning)", None),
    "Qwen3.5-122B-A10B": aai_index_dict.get("Qwen3.5 122B A10B (Reasoning)", None),
    "Qwen3.5-35B-A3B": aai_index_dict.get("Qwen3.5 35B A3B (Reasoning)", None),
    "Qwen3-VL-235B-A22B": aai_index_dict.get("Qwen3 VL 235B A22B (Reasoning)", None),
    "Qwen3-Next-80B-A3B": aai_index_dict.get("Qwen3 Next 80B A3B (Reasoning)", None),
    "Qwen3-VL-30B-A3B": aai_index_dict.get("Qwen3 VL 30B A3B (Reasoning)", None),

    # Kimi
    "Kimi-K2.6": aai_index_dict.get("Kimi K2.6", None),
    "Kimi-K2.5": aai_index_dict.get("Kimi K2.5 (Reasoning)", None),
	
    # Zhipu / GLM
    "GLM-5.1": aai_index_dict.get("GLM-5.1 (Reasoning)", None),
    "GLM-4.7": aai_index_dict.get("GLM-4.7 (Reasoning)", None),
    "GLM-5": aai_index_dict.get("GLM-5 (Reasoning)", None),

    # DeepSeek
    "DeepSeek-V4-Pro": aai_index_dict.get("DeepSeek V4 Pro (Reasoning, High Effort)", None),
    "DeepSeek-V4-Flash": aai_index_dict.get("DeepSeek V4 Flash (Reasoning, High Effort)", None),
}

df["aa_intelligence_index"] = df["model_name"].map(mapping)

In [ ]:
# ------------------------------------------------------------------
# CLEANING / PREP
# ------------------------------------------------------------------
plot_df = df.copy()

plot_df["accuracy_pct"] = plot_df["accuracy"] * 100

# Group key = manufacturer/family + version
plot_df["series"] = (
    plot_df["model_family"].astype(str)
    + " "
    + plot_df["model_version"].astype(str)
)

plot_df = plot_df.dropna(
    subset=["accuracy_pct", "aa_intelligence_index"]
).copy()

# Sort lines by model size if available, otherwise by intelligence index
plot_df["sort_key"] = np.where(
    plot_df["size"].notna(),
    plot_df["size"],
    plot_df["aa_intelligence_index"]
)

# ------------------------------------------------------------------
# STYLING
# ------------------------------------------------------------------
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(16, 10))

families = plot_df["model_family"].dropna().unique()
palette = sns.color_palette("tab10", n_colors=len(families))
family_colors = dict(zip(families, palette))

# ------------------------------------------------------------------
# DRAW CONNECTED SERIES
# ------------------------------------------------------------------
for series_name, g in plot_df.groupby("series"):
    g = g.sort_values("sort_key")

    fam = g["model_family"].iloc[0]
    color = family_colors.get(fam, "gray")

    # connected line
    ax.plot(
        g["aa_intelligence_index"],
        g["accuracy_pct"],
        lw=1.8,
        alpha=0.8,
        color=color,
        zorder=1
    )

    # scatter points
    ax.scatter(
        g["aa_intelligence_index"],
        g["accuracy_pct"],
        s=60,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.95,
        zorder=2
    )

# ------------------------------------------------------------------
# LABEL BEST POINT OF EACH SERIES (optional)
# ------------------------------------------------------------------
for series_name, g in plot_df.groupby("series"):
    best = g.loc[g["accuracy_pct"].idxmax()]

    ax.annotate(
        best["model_name"],
        xy=(best["aa_intelligence_index"], best["accuracy_pct"]),
        xytext=(8, 0),
        textcoords="offset points",
        fontsize=9,
        alpha=0.9
    )

# ------------------------------------------------------------------
# AXES
# ------------------------------------------------------------------
ax.set_xlabel("Artificial Analysis Intelligence Index", fontsize=14)
ax.set_ylabel("Accuracy (%)", fontsize=14)
ax.set_title(
    f"{eval_dir.upper()} - Model Performance vs AA Intelligence Index",
    fontsize=18,
    weight="bold",
    pad=20
)

# ax.set_ylim(0, 100)

# ------------------------------------------------------------------
# LEGEND (one entry per family)
# ------------------------------------------------------------------
handles = []
labels = []

for fam in families:
    h = plt.Line2D(
        [0], [0],
        color=family_colors[fam],
        marker="o",
        lw=2,
        markersize=7
    )
    handles.append(h)
    labels.append(fam)

ax.legend(
    handles,
    labels,
    title="Model Family",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=True
)

# ------------------------------------------------------------------
# GRID / LAYOUT
# ------------------------------------------------------------------
ax.grid(True, which="major", alpha=0.25)
ax.grid(True, which="minor", alpha=0.08)

plt.tight_layout()
plt.show()